In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q datasets transformers torchaudio pydub librosa soundfile jiwer
!apt-get install -q ffmpeg

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


In [ ]:
import pandas as pd

SHEET_ID = '1bujiO2NgtHlgqPlNvYAQf5_7ZcXARlIfNX5HNb9f8cI'
GID = '1786138861'
csv_url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}'

df = pd.read_csv(csv_url)
print(f'Total recordings: {len(df)}')
print(f'Total duration: {df["duration"].sum() / 3600:.2f} hours')
df.head()

Total recordings: 104
Total duration: 21.89 hours


,user_id,recording_id,language,duration,rec_url_gcp,transcription_url_gcp,metadata_url_gcp
0,245746,825780,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
1,291038,825727,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
2,246004,988596,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
3,93626,990175,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
4,286851,526266,hi,522,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...


In [ ]:

def fix_url(url):
    return url.replace(
        'https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/',
        'https://storage.googleapis.com/upload_goai/'
    )

df['transcription_url_gcp'] = df['transcription_url_gcp'].apply(fix_url)
df['rec_url_gcp'] = df['rec_url_gcp'].apply(fix_url)
df['metadata_url_gcp'] = df['metadata_url_gcp'].apply(fix_url)

# Verify
print(df['transcription_url_gcp'].iloc[0])


https://storage.googleapis.com/upload_goai/967179/825780_transcription.json


In [ ]:
import requests
import json
from tqdm import tqdm

def fetch_transcription(url):
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f'Failed {url}: {e}')
        return None

segments = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    transcription = fetch_transcription(row['transcription_url_gcp'])
    if transcription is None:
        continue
    for seg in transcription:
        segments.append({
            'recording_id': row['recording_id'],
            'user_id': row['user_id'],
            'rec_url_gcp': row['rec_url_gcp'],
            'start': seg['start'],
            'end': seg['end'],
            'speaker_id': seg.get('speaker_id', None),
            'text': seg['text'],
            'duration_seg': seg['end'] - seg['start']
        })

seg_df = pd.DataFrame(segments)
print(f'Total segments: {len(seg_df)}')
seg_df.head()

100%|██████████| 104/104 [00:38<00:00,  2.70it/s]

Total segments: 5941


,recording_id,user_id,rec_url_gcp,start,end,speaker_id,text,duration_seg
0,825780,245746,https://storage.googleapis.com/upload_goai/967...,0.11,14.42,245746,अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...,14.31
1,825780,245746,https://storage.googleapis.com/upload_goai/967...,14.42,29.03,245746,अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नह...,14.61
2,825780,245746,https://storage.googleapis.com/upload_goai/967...,29.03,41.84,245746,जंगल का सफर होता है जब हम रहने के लिए गए थे ना...,12.81
3,825780,245746,https://storage.googleapis.com/upload_goai/967...,42.47,50.57,245746,पहली बारी था क्योंकि चलना नहीं आता न वहाँ का ज...,8.10
4,825780,245746,https://storage.googleapis.com/upload_goai/967...,52.70,66.83,245746,हां तो फिर वहां जो दिन भर तो दिन में तो खोजने ...,14.13


In [ ]:
original_count = len(seg_df)

# Remove empty transcriptions
seg_df = seg_df[seg_df['text'].str.strip().str.len() > 0]

# Remove segments outside 1–30 second range
seg_df = seg_df[(seg_df['duration_seg'] >= 1.0) & (seg_df['duration_seg'] <= 30.0)]

# Remove single-character transcriptions (likely noise)
seg_df = seg_df[seg_df['text'].str.strip().str.len() > 1]

seg_df = seg_df.reset_index(drop=True)
print(f'Before filtering: {original_count} segments')
print(f'After filtering:  {len(seg_df)} segments')
print(f'Removed: {original_count - len(seg_df)} segments')

# Distribution of segment durations
print('\nSegment duration stats:')
print(seg_df['duration_seg'].describe())

Before filtering: 5941 segments
After filtering:  4924 segments
Removed: 1017 segments

Segment duration stats:
count    4924.000000
mean        8.896852
std         4.861056
min         1.020000
25%         3.960000
50%         9.765000
75%        13.710000
max        15.000000
Name: duration_seg, dtype: float64


In [ ]:
import unicodedata
import re

def normalize_text(text):
    # Unicode NFC normalization (important for Devanagari)
    text = unicodedata.normalize('NFC', text)
    # Collapse multiple whitespace
    text = re.sub(r'\s+', ' ', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text

seg_df['text_normalized'] = seg_df['text'].apply(normalize_text)

# Show a few examples
for i in range(3):
    print(f'Original:   {seg_df["text"].iloc[i]}')
    print(f'Normalized: {seg_df["text_normalized"].iloc[i]}')
    print()

Original:   अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब
Normalized: अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब

Original:   अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नहीं हो सकती थी तो हम वहां गया थे कुड़रमा घाटी तरफ पर दिवोग काफी जंगली एरिया है वह जो खांड जनजाति पाए जाती ना वहां पाए जाती है तो
Normalized: अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नहीं हो सकती थी तो हम वहां गया थे कुड़रमा घाटी तरफ पर दिवोग काफी जंगली एरिया है वह जो खांड जनजाति पाए जाती ना वहां पाए जाती है तो

Original:   जंगल का सफर होता है जब हम रहने के लिए गए थे नातो चाहते के साथ जैसे हम वहाँ पहले एंटर किये थे तो पहले तो गिर गया थे लगड़ा के उपर से नीचे
Normaliz

In [ ]:
import os
import io
import torchaudio
import torch
from pydub import AudioSegment

os.makedirs('/content/audio_segments', exist_ok=True)
os.makedirs('/content/audio_cache', exist_ok=True)

TARGET_SR = 16000

def download_audio(url, cache_path):
    """Download audio to cache if not already cached."""
    if os.path.exists(cache_path):
        return True
    try:
        r = requests.get(url, timeout=30, stream=True)
        r.raise_for_status()
        with open(cache_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except Exception as e:
        print(f'Download failed {url}: {e}')
        return False

def slice_and_resample(audio_path, start_sec, end_sec, out_path):
    """Slice segment and resample to 16kHz mono."""
    try:
        audio = AudioSegment.from_wav(audio_path)
        segment = audio[int(start_sec * 1000):int(end_sec * 1000)]
        segment = segment.set_frame_rate(TARGET_SR).set_channels(1)
        segment.export(out_path, format='wav')
        return True
    except Exception as e:
        print(f'Slice failed: {e}')
        return False

# Process — group by recording to avoid re-downloading
failed = []
seg_df['audio_path'] = None

for rec_id, group in tqdm(seg_df.groupby('recording_id')):
    rec_url = group['rec_url_gcp'].iloc[0]
    cache_path = f'/content/audio_cache/{rec_id}.wav'

    if not download_audio(rec_url, cache_path):
        failed.extend(group.index.tolist())
        continue

    for idx, row in group.iterrows():
        out_path = f'/content/audio_segments/{rec_id}_{idx}.wav'
        if slice_and_resample(cache_path, row['start'], row['end'], out_path):
            seg_df.at[idx, 'audio_path'] = out_path
        else:
            failed.append(idx)

print(f'\nFailed segments: {len(failed)}')
seg_df = seg_df[seg_df['audio_path'].notna()].reset_index(drop=True)
print(f'Usable segments: {len(seg_df)}')

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
100%|██████████| 104/104 [11:24<00:00,  6.58s/it]


Failed segments: 0
Usable segments: 4924


In [ ]:
from datasets import Dataset, Audio

hf_data = Dataset.from_dict({
    'audio': seg_df['audio_path'].tolist(),
    'sentence': seg_df['text_normalized'].tolist(),
    'recording_id': seg_df['recording_id'].tolist(),
    'speaker_id': seg_df['speaker_id'].tolist(),
})

# Cast audio column to HuggingFace Audio type (auto-resamples to 16kHz on access)
hf_data = hf_data.cast_column('audio', Audio(sampling_rate=TARGET_SR))

print(hf_data)
print('\nSample entry:')
print(hf_data[0]['sentence'])

Dataset({
    features: ['audio', 'sentence', 'recording_id', 'speaker_id'],
    num_rows: 4924
})

Sample entry:
अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब


In [ ]:
# Speaker-aware split: hold out ~10% of unique speakers for validation
import numpy as np

unique_speakers = seg_df['speaker_id'].unique()
np.random.seed(42)
val_speakers = set(np.random.choice(unique_speakers, size=max(1, int(len(unique_speakers) * 0.1)), replace=False))

train_idx = seg_df[~seg_df['speaker_id'].isin(val_speakers)].index.tolist()
val_idx   = seg_df[seg_df['speaker_id'].isin(val_speakers)].index.tolist()

train_ds = hf_data.select(train_idx)
val_ds   = hf_data.select(val_idx)

print(f'Train: {len(train_ds)} segments')
print(f'Val:   {len(val_ds)} segments')
print(f'Val speakers: {len(val_speakers)} out of {len(unique_speakers)}')

Train: 4491 segments
Val:   433 segments
Val speakers: 10 out of 102


In [ ]:
from transformers import WhisperProcessor

MODEL_ID = 'openai/whisper-small'
processor = WhisperProcessor.from_pretrained(MODEL_ID, language='Hindi', task='transcribe')

def prepare_dataset(batch):
    audio = batch['audio']
    # Compute log-mel spectrogram
    batch['input_features'] = processor.feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate']
    ).input_features[0]
    # Tokenize labels
    batch['labels'] = processor.tokenizer(batch['sentence']).input_ids
    return batch

train_ds = train_ds.map(prepare_dataset, remove_columns=train_ds.column_names, num_proc=1)
val_ds   = val_ds.map(prepare_dataset, remove_columns=val_ds.column_names, num_proc=1)

print('Preprocessing complete!')
print(train_ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/4491 [00:00<?, ? examples/s]

Map:   0%|          | 0/433 [00:00<?, ? examples/s]

Preprocessing complete!
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 4491
})


In [ ]:
train_ds.save_to_disk('/content/drive/MyDrive/josh_talks_asr/train_ds')
val_ds.save_to_disk('/content/drive/MyDrive/josh_talks_asr/val_ds')
seg_df.to_csv('/content/drive/MyDrive/josh_talks_asr/segments_metadata.csv', index=False)

print('Saved to Google Drive!')


Saving the dataset (0/9 shards):   0%|          | 0/4491 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/433 [00:00<?, ? examples/s]

Saved to Google Drive!


# FINE TUNING

In [ ]:
!pip install -q transformers datasets accelerate jiwer tensorboard evaluate
!apt-get install -q ffmpeg

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import torch

MODEL_ID = 'openai/whisper-small'

processor = WhisperProcessor.from_pretrained(MODEL_ID, language='Hindi', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

model.generation_config.language = 'hindi'
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None

print(f'Model parameters: {model.num_parameters() / 1e6:.1f}M')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU — switch to T4!"}')

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model parameters: 241.7M
Device: GPU


In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print('Data collator ready')

Data collator ready


In [ ]:
import evaluate
import numpy as np

metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer}

print('Metric ready')

Metric ready


In [ ]:
from datasets import load_from_disk

train_ds = load_from_disk('/content/drive/MyDrive/josh_talks_asr/train_ds')
val_ds   = load_from_disk('/content/drive/MyDrive/josh_talks_asr/val_ds')

print(f'Train: {len(train_ds)} segments')
print(f'Val:   {len(val_ds)} segments')

Train: 4491 segments
Val:   433 segments


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir='/content/drive/MyDrive/josh_talks_asr/whisper-small-hindi',
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000,
    gradient_checkpointing=True,
    per_device_eval_batch_size=8,
    eval_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    fp16=True,
    report_to='tensorboard',
    save_total_limit=2,
)

print('Training args ready')

Training args ready


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print('Starting training...')
trainer.train()

Starting training...


Step,Training Loss,Validation Loss,Wer
200,0.422412,0.404289,40.476858
400,0.259311,0.339441,37.849462
600,0.201386,0.328587,35.268817


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Step,Training Loss,Validation Loss,Wer
200,0.422412,0.404289,40.476858
400,0.259311,0.339441,37.849462
600,0.201386,0.328587,35.268817
800,0.172210,0.323956,35.895278
1000,0.134518,0.329790,35.334268


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=1000, training_loss=0.3099338507652283, metrics={'train_runtime': 8268.1385, 'train_samples_per_second': 1.935, 'train_steps_per_second': 0.121, 'total_flos': 4.6130376241152e+18, 'train_loss': 0.3099338507652283, 'epoch': 3.5587188612099645})

In [ ]:
trainer.save_model('/content/drive/MyDrive/josh_talks_asr/whisper-small-hindi-final')
processor.save_pretrained('/content/drive/MyDrive/josh_talks_asr/whisper-small-hindi-final')
print('Model saved!')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [ ]:
!pip install -q datasets --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 916.8 kB/s eta 0:00:00


In [ ]:
import requests
import json

# Direct FLEURS Hindi test data from HuggingFace
url = "https://huggingface.co/datasets/google/fleurs/resolve/main/data/hi_in/test.tsv"
r = requests.get(url)
print(r.status_code)

200


In [ ]:
import evaluate
metric = evaluate.load('wer')

# Evaluate fine-tuned model on our validation set
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

finetuned_pipe = pipeline(
    'automatic-speech-recognition',
    model='/content/drive/MyDrive/josh_talks_asr/whisper-small-hindi-final',
    generate_kwargs={'language': 'hindi', 'task': 'transcribe'},
    device=device
)

baseline_pipe = pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-small',
    generate_kwargs={'language': 'hindi', 'task': 'transcribe'},
    device=device
)

print('Both models loaded')

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Both models loaded


In [ ]:
import librosa
import torch
import numpy as np

baseline_preds = []
finetuned_preds = []
references = []

# Load both models directly
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained('openai/whisper-small', language='Hindi', task='transcribe')

baseline_model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small').to('cuda')
finetuned_model = WhisperForConditionalGeneration.from_pretrained(
    '/content/drive/MyDrive/josh_talks_asr/whisper-small-hindi-final'
).to('cuda')

def transcribe(model, audio):
    inputs = processor.feature_extractor(audio, sampling_rate=16000, return_tensors='pt').to('cuda')
    with torch.no_grad():
        predicted_ids = model.generate(
            inputs.input_features,
            language='hi',
            task='transcribe'
        )
    return processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]

for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    audio_path = row['audio_path']
    if not os.path.exists(audio_path):
        continue
    audio, _ = librosa.load(audio_path, sr=16000)
    baseline_preds.append(transcribe(baseline_model, audio))
    finetuned_preds.append(transcribe(finetuned_model, audio))
    references.append(row['text_normalized'].strip())

print(f'Evaluated: {len(references)} samples')
print(f'Sample baseline pred: {baseline_preds[0]}')
print(f'Sample finetuned pred: {finetuned_preds[0]}')
print(f'Sample reference:     {references[0]}')

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

100%|██████████| 433/433 [45:13<00:00,  6.27s/it]

Evaluated: 433 samples
Sample baseline pred:  अगर तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो �
Sample finetuned pred: काफी अच्छे होता है क्योंकि उनकी जन संखियां बहुत कमती जा रही है तो हमें उनको देखना था देखना था मतलब वो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाए जाती अ उधर की उधर की एरिया में उसके बारे में देखना और
Sample reference:     अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब


In [ ]:
!pip install -q transformers datasets accelerate jiwer tensorboard evaluate librosa
!apt-get install -q ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.1 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [ ]:
import requests
import os
from pydub import AudioSegment
from tqdm import tqdm

os.makedirs('/content/audio_segments', exist_ok=True)
os.makedirs('/content/audio_cache', exist_ok=True)

TARGET_SR = 16000

def download_audio(url, cache_path):
    if os.path.exists(cache_path):
        return True
    try:
        r = requests.get(url, timeout=30, stream=True)
        r.raise_for_status()
        with open(cache_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except Exception as e:
        print(f'Download failed: {e}')
        return False

def slice_and_resample(audio_path, start_sec, end_sec, out_path):
    try:
        audio = AudioSegment.from_wav(audio_path)
        segment = audio[int(start_sec * 1000):int(end_sec * 1000)]
        segment = segment.set_frame_rate(TARGET_SR).set_channels(1)
        segment.export(out_path, format='wav')
        return True
    except Exception as e:
        return False

# Only download val recordings to save time
val_rec_ids = val_df['recording_id'].unique()
print(f'Downloading {len(val_rec_ids)} recordings for val set...')

for rec_id, group in tqdm(val_df.groupby('recording_id')):
    rec_url = group['rec_url_gcp'].iloc[0]
    cache_path = f'/content/audio_cache/{rec_id}.wav'

    if not download_audio(rec_url, cache_path):
        continue

    for idx, row in group.iterrows():
        out_path = row['audio_path']
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        slice_and_resample(cache_path, row['start'], row['end'], out_path)

print('Done!')
print(f'Files created: {len(os.listdir("/content/audio_segments"))}')

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


100%|██████████| 10/10 [02:29<00:00, 14.90s/it]

Done!
Files created: 433


In [ ]:
# Evaluate on just 50 samples
val_df_small = val_df.sample(50, random_state=42).reset_index(drop=True)

baseline_preds = []
finetuned_preds = []
references = []

for _, row in tqdm(val_df_small.iterrows(), total=len(val_df_small)):
    if not os.path.exists(row['audio_path']):
        continue
    audio, _ = librosa.load(row['audio_path'], sr=16000)
    baseline_preds.append(transcribe(baseline_model, audio))
    finetuned_preds.append(transcribe(finetuned_model, audio))
    references.append(row['text_normalized'].strip())

baseline_wer  = 100 * metric.compute(predictions=baseline_preds, references=references)
finetuned_wer = 100 * metric.compute(predictions=finetuned_preds, references=references)

print(f'\n===== WER Results (50 samples) =====')
print(f'Baseline  (whisper-small pretrained): {baseline_wer:.2f}%')
print(f'Fine-tuned (whisper-small + Hindi data): {finetuned_wer:.2f}%')
print(f'Improvement: {baseline_wer - finetuned_wer:.2f}%')

100%|██████████| 50/50 [45:05<00:00, 54.11s/it]



===== WER Results (50 samples) =====
Baseline  (whisper-small pretrained): 230.11%
Fine-tuned (whisper-small + Hindi data): 41.00%
Improvement: 189.10%


In [ ]:
# Get all predictions and references from the 50 samples we already have
error_samples = []

for i, (bp, fp, ref) in enumerate(zip(baseline_preds, finetuned_preds, references)):
    # Only keep samples where fine-tuned model made errors
    if fp.strip() != ref.strip():
        error_samples.append({
            'index': i,
            'reference': ref,
            'finetuned_output': fp,
            'baseline_output': bp,
        })

print(f'Total error samples: {len(error_samples)}')

# Systematic sampling - every Nth error to get 25
step = max(1, len(error_samples) // 25)
sampled_errors = error_samples[::step][:25]

print(f'Sampled: {len(sampled_errors)} errors')
print(f'Sampling strategy: every {step}th error')

# Print all 25
for i, e in enumerate(sampled_errors):
    print(f'\n--- Error {i+1} ---')
    print(f'Reference:       {e["reference"]}')
    print(f'Fine-tuned output: {e["finetuned_output"]}')

Total error samples: 46
Sampled: 25 errors
Sampling strategy: every 1th error

--- Error 1 ---
Reference:       हाँ तो अ उनके जीवन के बारे में कुछ पहलू बता सकते हैं
Fine-tuned output: हां तो उनके जीवन के बारे में कुछ पहलू बता सकते हैं

--- Error 2 ---
Reference:       और अभी भी बोहोत से लोग आते हैं वहां घूमने फिरने, फोटो लेने,मतलब वो शिवलिंग अपने आपने ही एक बोहोत अद्भुत दृश्य है थोड़ा सा आगे जाने के बाद वहां पर एक पार्वती मंदिर है
Fine-tuned output: और अभी भी बहुत से लोग आते हैं वहां घूमने फिरने फोटो लेने वो वो श्यूगलिंग अपने आप में एक बहुत अध्वूद द्रश है थोड़ा सा आगे जाने के बाद वहां पर एक पार्वती मंदिर है

--- Error 3 ---
Reference:       कहीं ऐसा कांनटेक्ट वाला सीन है तो आप अपना भी बना सकते हो कि क्योंकि कल को सही में कोई कर सकता है आपके खिलाफ ऐसे लीगल नोटिस में क्योंकि वो तो पैसे वाली पार्टी है तो कहीं ना कहीं आपको दवा सकती है और आप दब जाओगे कहीं ना कहीं उन्होंने गवर्नमेंट।
Fine-tuned output: कहीं ऐसा कॉन्टेक्ट ओलासीन है तो आप अपनों भी बना सकते हो कि क्योंकि कल को सही में कोई कर सक

### Part G

In [ ]:
def transcribe_with_penalty(model, audio):
    inputs = processor.feature_extractor(audio, sampling_rate=16000, return_tensors='pt')
    with torch.no_grad():
        predicted_ids = model.generate(
            inputs.input_features,
            language='hi',
            task='transcribe',
            repetition_penalty=1.3,
            no_repeat_ngram_size=3
        )
    return processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]

# Test on the 5 most interesting error samples
test_indices = [7, 1, 4, 21, 23]  # errors 8, 2, 5, 22, 24 from our list

print("===== Before vs After Repetition Penalty =====\n")
for i in test_indices:
    row = val_df_small.iloc[i]
    if not os.path.exists(row['audio_path']):
        print(f"File missing: {row['audio_path']}")
        continue
    audio, _ = librosa.load(row['audio_path'], sr=16000)

    before = finetuned_preds[i] if i < len(finetuned_preds) else transcribe(finetuned_model, audio)
    after  = transcribe_with_penalty(finetuned_model, audio)
    ref    = row['text_normalized']

    print(f"Reference: {ref}")
    print(f"Before:    {before}")
    print(f"After:     {after}")
    print()

===== Before vs After Repetition Penalty =====



NameError: name 'val_df_small' is not defined